# Soil Organic Content Prediction — Improved Model

**Improvements over baseline:**
- Log-transform target (skewness 2.0 → 0.14)
- Feature engineering: spectral aggregates, missingness flags, cation/particle ratios
- LightGBM + XGBoost ensemble
- Better hyperparameters (lower LR, regularization, bagging)
- Drop `sampling_depth_cm` (only 1 unique value)


In [9]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold


In [10]:
train = pd.read_csv('dataset/train.csv')
test  = pd.read_csv('dataset/test.csv')
print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head(3)


Train: (11210, 52)  |  Test: (2670, 51)


,sample_id,source_id,has_band_A_spectrum,has_band_B_spectrum,sampling_strategy,sampling_depth_cm,geo_zone_macro,geo_zone_micro,geo_zone_meso,land_cover_type,...,spectral_band_B_PC_6,spectral_band_B_PC_7,spectral_band_B_PC_8,spectral_band_B_PC_9,spectral_band_B_PC_10,spectral_band_B_PC_11,spectral_band_B_PC_12,spectral_band_B_PC_13,spectral_band_B_PC_14,spectral_band_B_PC_15
0,train_00001,Source_01,YES,NO,Auger,0-20,SE,Unknown,State_01,Seasonal Semideciduous Forest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,train_00002,Source_10,YES,NO,Auger,0-20,MW,Loc_011,State_10,Savannah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,train_00003,Source_04,YES,NO,Auger,0-20,S,Loc_001,State_06,Unknown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Feature Engineering

In [11]:
def engineer(df):
    df = df.copy()

    # ── Boolean encoding ──────────────────────────────────────────────────────
    df['has_band_A_spectrum'] = df['has_band_A_spectrum'].map({'YES': 1, 'NO': 0})
    df['has_band_B_spectrum'] = df['has_band_B_spectrum'].map({'YES': 1, 'NO': 0})

    # ── Missingness flags ─────────────────────────────────────────────────────
    # High-NA columns carry signal in whether the value exists
    df['lat_missing']   = df['latitude'].isna().astype(int)
    df['bandB_missing'] = df['spectral_band_B_PC_1'].isna().astype(int)
    df['acid_missing']  = df['property_acidity_index'].isna().astype(int)

    # ── Spectral band aggregates ──────────────────────────────────────────────
    band_a = [c for c in df.columns if c.startswith('spectral_band_A_PC_')]
    band_b = [c for c in df.columns if c.startswith('spectral_band_B_PC_')]

    df['bandA_mean']  = df[band_a].mean(axis=1)
    df['bandA_std']   = df[band_a].std(axis=1)
    df['bandA_range'] = df[band_a].max(axis=1) - df[band_a].min(axis=1)
    df['bandA_l2']    = np.sqrt((df[band_a] ** 2).sum(axis=1))

    df['bandB_mean']  = df[band_b].mean(axis=1)
    df['bandB_std']   = df[band_b].std(axis=1)
    df['bandB_range'] = df[band_b].max(axis=1) - df[band_b].min(axis=1)

    # ── Cation & particle features ────────────────────────────────────────────
    df['cation_ratio']   = df['cation_Ca'] / (df['cation_Mg'] + 1e-6)
    df['cation_sum']     = df['cation_Ca'].fillna(0) + df['cation_Mg'].fillna(0)
    df['particle_sum']   = df['property_particle_coarse'].fillna(0) + df['property_particle_fine'].fillna(0)
    df['particle_ratio'] = df['property_particle_coarse'] / (df['property_particle_fine'] + 1e-6)

    return df

train = engineer(train)
test  = engineer(test)
print('Features created!')


Features created!


## Prepare X / y

In [12]:
cat_cols = [
    'geo_zone_macro', 'geo_zone_micro', 'geo_zone_meso',
    'land_cover_type', 'biome', 'parent_rock_type',
    'source_id', 'sampling_strategy'
]
drop_cols = [
    'sample_id',
    'cation_Na',          # 96% missing
    'sampling_depth_cm',  # only 1 unique value → no signal
    'property_organic_content'
]

X      = train.drop(columns=[c for c in drop_cols if c in train.columns])
y      = train['property_organic_content']

# Log-transform: skewness 2.0 → 0.14 → improves tree splits
y_log  = np.log1p(y)

X_test = test.drop(columns=[c for c in drop_cols if c in test.columns])

for col in cat_cols:
    X[col]     = X[col].astype('category')
    X_test[col]= X_test[col].astype('category')

print(f'Features: {X.shape[1]}  |  Train rows: {len(X)}  |  Test rows: {len(X_test)}')


Features: 62  |  Train rows: 11210  |  Test rows: 2670


## LightGBM with 5-Fold CV

In [13]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgb_params = {
    'objective':         'regression',
    'metric':            'rmse',
    'boosting_type':     'gbdt',
    'learning_rate':     0.02,
    'num_leaves':        63,
    'min_child_samples': 20,
    'feature_fraction':  0.75,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'lambda_l1':         0.1,
    'lambda_l2':         0.5,
    'verbose':           -1,
    'random_state':      42,
}

lgb_oof  = np.zeros(len(X))
lgb_test = np.zeros(len(X_test))
lgb_rmse = []

for fold, (tr, val) in enumerate(kf.split(X, y_log)):
    Xtr, Xval = X.iloc[tr], X.iloc[val]
    ytr, yval = y_log.iloc[tr], y_log.iloc[val]

    d_tr  = lgb.Dataset(Xtr, label=ytr, categorical_feature=cat_cols)
    d_val = lgb.Dataset(Xval, label=yval, categorical_feature=cat_cols, reference=d_tr)

    m = lgb.train(
        lgb_params, d_tr,
        num_boost_round=3000,
        valid_sets=[d_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(period=200)
        ]
    )

    pred             = np.expm1(m.predict(Xval, num_iteration=m.best_iteration))
    lgb_oof[val]     = pred
    lgb_test        += np.expm1(m.predict(X_test, num_iteration=m.best_iteration)) / 5

    rmse = np.sqrt(mean_squared_error(y.iloc[val], pred))
    lgb_rmse.append(rmse)
    print(f'Fold {fold+1}  RMSE = {rmse:.4f}  (best iter={m.best_iteration})')

print(f'\nLightGBM CV RMSE: {np.mean(lgb_rmse):.4f} ± {np.std(lgb_rmse):.4f}')


Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.298774
[400]	valid_0's rmse: 0.291168
[600]	valid_0's rmse: 0.289593
[800]	valid_0's rmse: 0.288672
[1000]	valid_0's rmse: 0.28815
Early stopping, best iteration is:
[1080]	valid_0's rmse: 0.288045
Fold 1  RMSE = 11.0907  (best iter=1080)
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.298789
[400]	valid_0's rmse: 0.287792
[600]	valid_0's rmse: 0.285177
[800]	valid_0's rmse: 0.283998
[1000]	valid_0's rmse: 0.283014
[1200]	valid_0's rmse: 0.282554
[1400]	valid_0's rmse: 0.282342
Early stopping, best iteration is:
[1430]	valid_0's rmse: 0.282279
Fold 2  RMSE = 10.9016  (best iter=1430)
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.294412
[400]	valid_0's rmse: 0.286336
[600]	valid_0's rmse: 0.284545
[800]	valid_0's rmse: 0.2839
[1000]	valid_0's rmse: 0.283338
[1200]	valid_0's rmse: 0.28312
Early stopping, best iteration is:
[12

## XGBoost with 5-Fold CV

In [14]:
# XGBoost doesn't support native categoricals — encode as codes
X_xgb      = X.copy()
X_test_xgb = X_test.copy()
for col in cat_cols:
    X_xgb[col]      = X_xgb[col].cat.codes
    X_test_xgb[col] = X_test_xgb[col].cat.codes

xgb_params = {
    'objective':        'reg:squarederror',
    'eval_metric':      'rmse',
    'learning_rate':    0.02,
    'max_depth':        6,
    'min_child_weight': 10,
    'subsample':        0.8,
    'colsample_bytree': 0.7,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'tree_method':      'hist',
    'random_state':     42,
    'verbosity':        0,
}

xgb_oof   = np.zeros(len(X))
xgb_test  = np.zeros(len(X_test))
xgb_rmse  = []

for fold, (tr, val) in enumerate(kf.split(X_xgb, y_log)):
    Xtr, Xval = X_xgb.iloc[tr], X_xgb.iloc[val]
    ytr, yval = y_log.iloc[tr], y_log.iloc[val]

    dtr  = xgb.DMatrix(Xtr, label=ytr)
    dval = xgb.DMatrix(Xval, label=yval)

    m = xgb.train(
        xgb_params, dtr,
        num_boost_round=3000,
        evals=[(dval, 'val')],
        early_stopping_rounds=100,
        verbose_eval=False,
    )

    pred             = np.expm1(m.predict(dval))
    xgb_oof[val]     = pred
    xgb_test        += np.expm1(m.predict(xgb.DMatrix(X_test_xgb))) / 5

    rmse = np.sqrt(mean_squared_error(y.iloc[val], pred))
    xgb_rmse.append(rmse)
    print(f'Fold {fold+1}  RMSE = {rmse:.4f}')

print(f'\nXGBoost CV RMSE: {np.mean(xgb_rmse):.4f} ± {np.std(xgb_rmse):.4f}')


Fold 1  RMSE = 11.4305
Fold 2  RMSE = 11.2299
Fold 3  RMSE = 11.6889
Fold 4  RMSE = 12.2058
Fold 5  RMSE = 12.8690

XGBoost CV RMSE: 11.8848 ± 0.5908


## Ensemble & Submission

In [15]:
# Weighted ensemble — try a few weights
for w in [0.5, 0.6, 0.7]:
    ens  = w * lgb_oof + (1 - w) * xgb_oof
    rmse = np.sqrt(mean_squared_error(y, ens))
    print(f'{w:.1f} LGB + {1-w:.1f} XGB  →  OOF RMSE = {rmse:.4f}')

# Final predictions (use best weight, e.g. 0.7 LGB)
W_LGB = 0.7
final_preds = W_LGB * lgb_test + (1 - W_LGB) * xgb_test

sub = test[['sample_id']].copy()
sub['property_organic_content'] = final_preds
sub.to_csv('vtry2.csv', index=False)
print('\nSubmission saved!')
sub.describe()


0.5 LGB + 0.5 XGB  →  OOF RMSE = 11.6719
0.6 LGB + 0.4 XGB  →  OOF RMSE = 11.6495
0.7 LGB + 0.3 XGB  →  OOF RMSE = 11.6348

Submission saved!


,property_organic_content
count,2670.000000
mean,32.974359
std,19.268131
min,5.815240
25%,19.450820
50%,26.975399
75%,41.742522
max,159.773825
